# Franken · Qwen1.7B · calibration

A100 40 GB. Запустите все ячейки. Предел — 2 часа с установкой. Код и данные внутри.

Две ветки обучения: 5 warmup + 20 шагов; проверка checkpoint в новом процессе. Затем 14 RU + 14 EN вопросов исходной модели, chat 5-shot. Не полный baseline.


In [ ]:
import base64
import hashlib
import json
import os
import signal
import subprocess
import sys
import threading
import time
import zipfile
import zlib
from pathlib import Path
from uuid import uuid4

from google.colab import files

SESSION_STARTED = time.time()
PAYLOAD = (
    '...'
    'H8uaHkQ='
)
SOURCES = json.loads(zlib.decompress(base64.b64decode(PAYLOAD)))
SOURCE_SHA256 = {
    'eval_support/__init__.py': (
        '78d7a894f2ba87185a30a183c3696b907b0b0e04f8ccfda7'
        'baca3c23c258d849'
    ),
    'eval_support/franken_smoke.py': (
        '3792dce6b33535fad8775120f572a677bf8c68e7cad1d046'
        'ea93bc6c5300e038'
    ),
    'eval_support/franken_tokenizer.py': (
        '6e36bd9df51198b888c913df300b4283e9a93a8e65a03b31'
        '042f8924cac077fc'
    ),
    'eval_support/franken_calibration.py': (
        'bbf53f71e7fa5d73fd0568033eb3956cfc37c41093eebc5b'
        '7c6af001ed6ddfe7'
    ),
    'eval_support/franken_calibration_tasks.py': (
        '8c4fbb18f3d5214302b6cacdc269e780c5cdbca94deccec7'
        'aa3edb3248687ca6'
    ),
    'assets/adapted/chat_template.jinja': (
        'f950548988e50713a9975a11be0eaf1d29c4fe4be0ba22a5'
        '7daeebbc27a7adb9'
    ),
    'assets/adapted/tokenizer.json': (
        'b54961c58f9688aa15364e7f7ef17cead30bfdc4d2c63b48'
        '6a34d4898de02b04'
    ),
    'assets/adapted/tokenizer_config.json': (
        '0fe2ab47b292d91fbbb57eda2c0f9528f7aba9e0f42c4dda'
        '9248ac3df5a28fb0'
    ),
    'assets/config.json': (
        'b99eea9a9d7ba5fb8488585215bfb219e930a9840d2d6cec'
        '4f3056e346f104a3'
    ),
    'assets/data_manifest.json': (
        '49fd1d29ecc48c58154af19ee5f2b2c1611385b98ff1aedc'
        '6bcb01463db5b52e'
    ),
    'assets/inputs.json': (
        '9729a001a64234f21726a49686782eaffee0d2c697dd53e6'
        '1468a913e3e75247'
    ),
    'assets/manifest.json': (
        '069edf486827500a0fbb6800f5ca65f5670d8b668614f78e'
        'cc1fbf1d7edb67ec'
    ),
    'assets/original/chat_template.jinja': (
        'f950548988e50713a9975a11be0eaf1d29c4fe4be0ba22a5'
        '7daeebbc27a7adb9'
    ),
    'assets/original/tokenizer.json': (
        'be75606093db2094d7cd20f3c2f385c212750648bd6ea4fb'
        '2bf507a6a4c55506'
    ),
    'assets/original/tokenizer_config.json': (
        '0fe2ab47b292d91fbbb57eda2c0f9528f7aba9e0f42c4dda'
        '9248ac3df5a28fb0'
    ),
    'assets/data_config.json': (
        '91848449f74ca4c4771dd39c4bbddf87723cc0ace0d16b28'
        '966fa27b3669b49c'
    ),
    'assets/data_README.md': (
        'dbc8ffac6581af19df155c427591a4a09a7c6f380dc4d230'
        '2cc83b1bd85c0dfc'
    ),
}
RUN_ROOT = Path("/content/lapa_franken_calibration") / uuid4().hex
RUN_ROOT.mkdir(parents=True)
RESULTS = RUN_ROOT / "results"
RESULTS.mkdir()

for name, content in SOURCES.items():
    if hashlib.sha256(content.encode()).hexdigest() != SOURCE_SHA256[name]:
        raise ValueError(f"Повреждён исходник: {name}")
    path = RUN_ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("x", encoding="utf-8") as stream:
        stream.write(content)

CONFIG_PATH = RUN_ROOT / "assets/config.json"
CONFIG = json.loads(CONFIG_PATH.read_text())
DEADLINE = SESSION_STARTED + CONFIG["budget_seconds"]
with (RESULTS / "session.json").open("x") as stream:
    json.dump({"started_unix": SESSION_STARTED, "deadline_unix": DEADLINE,
               "run_id": RUN_ROOT.name, "sources": SOURCE_SHA256}, stream)


def file_sha256(path):
    value = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            value.update(chunk)
    return value.hexdigest()


def export_results():
    stamp = uuid4().hex
    archive = RUN_ROOT / f"FRANKEN_CALIBRATION_{stamp}.zip"
    members = []
    for base in (RESULTS, RUN_ROOT / "eval_support", RUN_ROOT / "assets"):
        for path in base.rglob("*"):
            if path.is_file() and path.suffix in {
                    ".json", ".jsonl", ".log", ".py", ".jinja", ".md"}:
                members.append(path)
    manifest = {str(p.relative_to(RUN_ROOT)): file_sha256(p)
                for p in sorted(members)}
    with zipfile.ZipFile(archive, "x", zipfile.ZIP_DEFLATED) as output:
        for path in members:
            output.write(path, path.relative_to(RUN_ROOT))
        output.writestr("export_manifest.json", json.dumps({
            "files": manifest,
            "weights_included": False,
            "elapsed_seconds": time.time() - SESSION_STARTED,
        }, indent=2))
    print(f"ZIP: {archive}\nSHA-256: {file_sha256(archive)}")
    files.download(str(archive))
    return archive


def _run_process(command, log_name):
    remaining = DEADLINE - time.time() - 120
    if remaining <= 0:
        raise TimeoutError("Бюджет исчерпан; новый процесс не запущен")
    environment = os.environ.copy()
    environment["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    environment["PYTHONUNBUFFERED"] = "1"
    environment["TOKENIZERS_PARALLELISM"] = "false"
    environment["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
    with (RESULTS / log_name).open("x", encoding="utf-8") as log:
        process = subprocess.Popen(
            command, cwd=RUN_ROOT, env=environment, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1,
            start_new_session=True,
        )

        def stop():
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass

        timer = threading.Timer(remaining, stop)
        timer.daemon = True
        timer.start()
        try:
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end="", flush=True)
            code = process.wait()
            if code:
                raise RuntimeError(f"Процесс завершился с кодом {code}")
        finally:
            timer.cancel()
            if process.poll() is None:
                stop()
                process.wait()
            process.stdout.close()


print("Qwen3-1.7B-Base: calibration, предел 2 A100-часа.")
print("Две ветки: 5 warmup + 20 шагов; 14 RU + 14 EN вопросов.")
print(f"Результаты: {RESULTS}")


def run_process(command, log_name):
    started = time.monotonic()
    try:
        return _run_process(command, log_name)
    finally:
        path = RESULTS / (log_name + ".timing.json")
        with path.open("x") as stream:
            json.dump({"elapsed_seconds": time.monotonic() - started}, stream)


## Зависимости


In [ ]:
ENV_PYTHON = RUN_ROOT / "venv/bin/python"
PIP = [sys.executable, "-m", "pip", "--python", str(ENV_PYTHON)]
INSTALL_ID = uuid4().hex[:8]
try:
    if not (3, 11) <= sys.version_info[:2] <= (3, 13):
        raise RuntimeError("Поддерживается Python 3.11–3.13")
    print("Создаём окружение и устанавливаем зависимости.")
    run_process(
        [sys.executable, "-m", "venv", "--without-pip",
         str(RUN_ROOT / "venv")],
        f"venv_{INSTALL_ID}.log",
    )
    run_process(PIP + [
        "install", "--disable-pip-version-check",
        "torch==2.10.0", "transformers==5.3.0", "tokenizers==0.22.2",
        "huggingface-hub==1.5.0", "safetensors==0.8.0", "numpy==2.2.4",
        "accelerate==1.12.0", "datasets==4.6.1", "jinja2==3.1.6",
        "lm-eval @ git+https://github.com/EleutherAI/"
        "lm-evaluation-harness.git@"
        "d6de81643928d653435c431bae19945d41d32520",
    ], f"install_{INSTALL_ID}.log")
    run_process(
        PIP + ["check"], f"pip_check_{INSTALL_ID}.log")
    run_process(
        PIP + ["freeze"], f"environment_{INSTALL_ID}.log")
except BaseException:
    export_results()
    raise


## Замер

ZIP скачивается автоматически, включая частичный результат при ошибке. Веса остаются в среде. После скачивания отключите GPU. Drive и загрузка входного ZIP не нужны.


In [ ]:
try:
    stages = (
        "preflight", "language", "language_resume", "instruction",
        "instruction_resume", "evaluate", "summary",
    )
    for index, stage in enumerate(stages, 1):
        print(f"Этап {index}/{len(stages)}: {stage}", flush=True)
        run_process([
            str(ENV_PYTHON), "-m", "eval_support.franken_calibration",
            stage, str(RUN_ROOT / "assets"), str(RESULTS), str(DEADLINE),
        ], f"{stage}.log")
    summary = json.loads((RESULTS / "summary.json").read_text())
    print(summary["status"])
    for language, result in summary["evaluation"].items():
        print(language, result)
finally:
    export_results()
    print("Скачайте ZIP и отключите GPU. Полное обучение не запускалось.")
